**Usando as tabelas Silver criadas anteriormente:**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "workspace"
schema_silver = "ecommerce_silver"
schema_gold = "ecommerce_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_gold}")

clientes = spark.table(f"{catalogo}.{schema_silver}.clientes")
produtos = spark.table(f"{catalogo}.{schema_silver}.produtos")
pedidos = spark.table(f"{catalogo}.{schema_silver}.pedidos")
itens_pedido = spark.table(f"{catalogo}.{schema_silver}.itens_pedido")


**Criando a dimensão de clientes:**

A coluna sk_clienteé uma chave substituta criada para uso nos relacionamentos analíticos da camada Gold.



In [0]:
dim_cliente = (
    clientes
    .withColumn(
        "sk_cliente",
        F.xxhash64(F.col("customer_id"))
    )
    .select(
        "sk_cliente",
        "customer_id",
        "customer_name",
        "customer_age",
        "gender",
        "customer_segment",
        "customer_city",
        "customer_state",
        "customer_country",
        "region",
        "customer_postal_code",
        "customer_acquisition_cost"
    )
)


**Criando a dimensão de produtos:**

In [0]:
dim_produto = (
    produtos
    .withColumn(
        "sk_produto",
        F.xxhash64(F.col("product_id"))
    )
    .select(
        "sk_produto",
        "product_id",
        "product_name",
        "product_category",
        "product_subcategory",
        "brand",
        "supplier",
        "catalog_unit_price",
        "catalog_product_cost",
        "product_rating"
    )
)


**Criando a dimensão de dados:**

A dimensão dos dados será derivada da coluna order_dateda tabela Silver de pedidos.

In [0]:
dim_data = (
    pedidos
    .select("order_date")
    .filter(F.col("order_date").isNotNull())
    .dropDuplicates()
    .select(
        F.date_format("order_date", "yyyyMMdd")
        .cast("int")
        .alias("sk_data"),

        "order_date",

        F.year("order_date").alias("ano"),
        F.quarter("order_date").alias("trimestre"),
        F.month("order_date").alias("mes"),
        F.date_format("order_date", "MMMM").alias("nome_mes"),
        F.date_format("order_date", "yyyy-MM").alias("ano_mes"),
        F.dayofmonth("order_date").alias("dia_mes"),
        F.date_format("order_date", "EEEE").alias("dia_semana")
    )
)


Exemplo de uma chave de data:``

20231106


**Criando a dimensão comercial:**

In [0]:
dim_comercial = (
    pedidos
    .select(
        "sales_channel",
        "marketing_channel",
        "campaign_name",
        "coupon_code",
        "customer_type"
    )
    .dropDuplicates()
    .withColumn(
        "sk_comercial",
        F.xxhash64(
            F.coalesce(F.col("sales_channel"), F.lit("N/A")),
            F.coalesce(F.col("marketing_channel"), F.lit("N/A")),
            F.coalesce(F.col("campaign_name"), F.lit("N/A")),
            F.coalesce(F.col("coupon_code"), F.lit("N/A")),
            F.coalesce(F.col("customer_type"), F.lit("N/A"))
        )
    )
    .select(
        "sk_comercial",
        "sales_channel",
        "marketing_channel",
        "campaign_name",
        "coupon_code",
        "customer_type"
    )
)


**Criando a dimensão de pedidos:**

In [0]:
pedidos_com_chaves = (
    pedidos
    .withColumn(
        "sk_pedido",
        F.xxhash64(F.col("order_id"))
    )
    .withColumn(
        "sk_data",
        F.date_format("order_date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "sk_comercial",
        F.xxhash64(
            F.coalesce(F.col("sales_channel"), F.lit("N/A")),
            F.coalesce(F.col("marketing_channel"), F.lit("N/A")),
            F.coalesce(F.col("campaign_name"), F.lit("N/A")),
            F.coalesce(F.col("coupon_code"), F.lit("N/A")),
            F.coalesce(F.col("customer_type"), F.lit("N/A"))
        )
    )
)

dim_pedido = (
    pedidos_com_chaves
    .select(
        "sk_pedido",
        "order_id",
        "order_status",
        "payment_method",
        "payment_status",
        "currency",
        "shipping_method",
        "warehouse",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_status",
        "return_status",
        "return_reason",
        "customer_rating",
        "review_sentiment",
        "is_repeat_customer",
        "customer_order_count",
        "customer_lifetime_value",
        "loyalty_points_earned",
        "loyalty_points_redeemed"
    )
)


**Criando uma tabela de itens de pedido:**

Essa é a principal tabela analítica do projeto.

As métricas de receita, desconto, custo, quantidade e lucro serão obtidos de itens_pedido, pois essa tabela possui o detalhe por produto vendido.

In [0]:
janela_item = Window.partitionBy(
    "order_id",
    "product_id"
).orderBy(
    F.col("unit_price"),
    F.col("gross_sales"),
    F.col("net_sales")
)

fato_item_pedido = (
    itens_pedido
    .withColumn(
        "numero_item_pedido",
        F.row_number().over(janela_item)
    )
    .join(
        pedidos_com_chaves.select(
            "order_id",
            "customer_id",
            "sk_pedido",
            "sk_data",
            "sk_comercial"
        ),
        on="order_id",
        how="inner"
    )
    .withColumn(
        "sk_cliente",
        F.xxhash64(F.col("customer_id"))
    )
    .withColumn(
        "sk_produto",
        F.xxhash64(F.col("product_id"))
    )
    .withColumn(
        "sk_item_pedido",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("order_id"),
                F.col("product_id"),
                F.col("numero_item_pedido")
            ),
            256
        )
    )
    .withColumn(
        "profit_margin_percentage",
        F.round(
            F.when(
                F.col("net_sales") > 0,
                (F.col("profit") / F.col("net_sales")) * 100
            ),
            2
        )
    )
    .select(
        "sk_item_pedido",
        "sk_cliente",
        "sk_produto",
        "sk_data",
        "sk_pedido",
        "sk_comercial",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_percentage",
        "discount_amount",
        "gross_sales",
        "tax_amount",
        "shipping_cost",
        "net_sales",
        "product_cost",
        "profit",
        "profit_margin_percentage"
    )
)


**Criando uma tabela analítica agregada mensal:**

Esta tabela não substitui a tabela detalhada. Ela é uma tabela agregada complementar para análises mais rápidas de vendas mensais por categoria e canal.

In [0]:
fato_vendas_mensais = (
    fato_item_pedido.alias("f")
    .join(
        dim_data.alias("d"),
        F.col("f.sk_data") == F.col("d.sk_data"),
        "inner"
    )
    .join(
        dim_produto.alias("p"),
        F.col("f.sk_produto") == F.col("p.sk_produto"),
        "inner"
    )
    .join(
        dim_comercial.alias("c"),
        F.col("f.sk_comercial") == F.col("c.sk_comercial"),
        "inner"
    )
    .groupBy(
        F.col("d.ano_mes"),
        F.col("p.product_category"),
        F.col("c.sales_channel"),
        F.col("c.marketing_channel")
    )
    .agg(
        F.sum(F.col("f.quantity")).alias("quantidade_vendida"),
        F.round(F.sum(F.col("f.gross_sales")), 2).alias("venda_bruta"),
        F.round(F.sum(F.col("f.discount_amount")), 2).alias("desconto_total"),
        F.round(F.sum(F.col("f.net_sales")), 2).alias("receita_liquida"),
        F.round(F.sum(F.col("f.product_cost")), 2).alias("custo_total"),
        F.round(F.sum(F.col("f.profit")), 2).alias("lucro_total")
    )
)


**Salvando as tabelas Gold:**

In [0]:
tabelas_gold = {
    "dim_cliente": dim_cliente,
    "dim_produto": dim_produto,
    "dim_data": dim_data,
    "dim_pedido": dim_pedido,
    "dim_comercial": dim_comercial,
    "fato_item_pedido": fato_item_pedido,
    "fato_vendas_mensais": fato_vendas_mensais
}

for nome_tabela, dataframe in tabelas_gold.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalogo}.{schema_gold}.{nome_tabela}")
    )

    print(f"Tabela Gold criada: {catalogo}.{schema_gold}.{nome_tabela}")
    print(f"Quantidade de registros: {dataframe.count()}")


Tabela Gold criada: workspace.ecommerce_gold.dim_cliente
Quantidade de registros: 25000
Tabela Gold criada: workspace.ecommerce_gold.dim_produto
Quantidade de registros: 1175
Tabela Gold criada: workspace.ecommerce_gold.dim_data
Quantidade de registros: 1826
Tabela Gold criada: workspace.ecommerce_gold.dim_pedido
Quantidade de registros: 138116
Tabela Gold criada: workspace.ecommerce_gold.dim_comercial
Quantidade de registros: 5815
Tabela Gold criada: workspace.ecommerce_gold.fato_item_pedido
Quantidade de registros: 397569
Tabela Gold criada: workspace.ecommerce_gold.fato_vendas_mensais
Quantidade de registros: 34815


**Validando a criação das tabelas:**

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_gold;


database,tableName,isTemporary
ecommerce_gold,dim_cliente,false
ecommerce_gold,dim_comercial,false
ecommerce_gold,dim_data,false
ecommerce_gold,dim_pedido,false
ecommerce_gold,dim_produto,false
ecommerce_gold,fato_item_pedido,false
ecommerce_gold,fato_vendas_mensais,false


**Conferindo a estrutura:**

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.fato_item_pedido;


col_name,data_type,comment
sk_item_pedido,string,null
sk_cliente,bigint,null
sk_produto,bigint,null
sk_data,int,null
sk_pedido,bigint,null
sk_comercial,bigint,null
order_id,string,null
product_id,string,null
quantity,int,null
unit_price,"decimal(18,2)",null


Visualizando alguns registros:

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.fato_item_pedido
LIMIT 10;


sk_item_pedido,sk_cliente,sk_produto,sk_data,sk_pedido,sk_comercial,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage
378a116dbe88e3e4c1bec293d01029ffc165e52b102b321887de54e2b73cf16e,3541997667526014623,9193282904692646284,20221026,8903629385399243214,7626912671941354127,ORD-100002,PROD-000132,1,643.49,0.000000,0.00,643.49,32.17,31.75,707.41,312.22,363.44,51.38
76019b3d9794860410c3243759280f953776a265330de3c4a650f2d34c9f6434,3541997667526014623,3714893824816619141,20221026,8903629385399243214,7626912671941354127,ORD-100002,PROD-000730,4,233.33,0.000000,0.00,933.32,46.67,5.28,985.27,510.64,469.35,47.64
7c44f659a0ca08b68e49172978e8502d7c879181004b2588c4fe2919eb7a408a,-3139848357791111567,6172634258489261608,20250329,-1258966004084791145,-1416841444318546326,ORD-100004,PROD-000475,1,12.91,0.094303,1.22,12.91,2.10,10.10,23.89,4.26,9.53,39.89
723a3dacc0db21f3e81f10c57f617e67c4ae841a1b1b782eeeb82feeb71f73ce,-3139848357791111567,-7137199947328216364,20250329,-1258966004084791145,-1416841444318546326,ORD-100004,PROD-000782,1,116.11,0.098960,11.49,116.11,18.83,2.30,125.75,67.97,55.48,44.12
7a65225bb236aef0bbb387bc7afd7205aefd68d4d74861d07ac06db47633269d,-3139848357791111567,-313634662720282612,20250329,-1258966004084791145,-1416841444318546326,ORD-100004,PROD-000912,1,335.86,0.042021,14.11,335.86,57.92,4.06,383.73,195.75,183.92,47.93
4f00f43811b5b786507800f0090e1424b8b823002d9cbb44b787e7f089676aca,-6226880087847751051,2700813565094814040,20230307,-8813521586098746864,-3146606341713494887,ORD-100016,PROD-000319,2,176.23,0.180713,63.69,352.46,20.21,3.80,312.78,157.12,151.86,48.55
fda9a9312c6cfe79fcc72d747877f240c6a345024c89b4ff53c633a98870e71c,-6226880087847751051,160423038422858089,20230307,-8813521586098746864,-3146606341713494887,ORD-100016,PROD-001037,1,61.00,0.090087,5.50,61.00,3.88,15.15,74.53,24.60,34.78,46.67
7df5ccf5042888d2d640ec83ed5fc7e18ac20b2bc41cb4f8ae69c255d861293b,5932462398612812751,3469000970221866109,20241227,4637041998314709959,-2373374872675047471,ORD-100022,PROD-000381,2,416.72,0.186442,155.39,833.44,88.15,17.74,783.94,378.66,387.54,49.43
a960f5cb7e319fb0880bbe841986bb167ee0e089df87a12e0eea5c477093e490,5932462398612812751,7632619664919627041,20241227,4637041998314709959,-2373374872675047471,ORD-100022,PROD-000717,3,155.85,0.449832,210.32,467.55,33.44,27.62,318.29,260.46,30.21,9.49
09249886ba2ffce5167796f6e8ed9d70732cbdcaffa1d3a0ab4ad9599bb518f9,5932462398612812751,-7271644652052874124,20241227,4637041998314709959,-2373374872675047471,ORD-100022,PROD-001140,1,199.76,0.153306,30.62,199.76,21.99,28.25,219.38,71.51,119.62,54.53


**Conferindo o total das tabelas:**

In [0]:
%sql
SELECT
    'dim_cliente' AS tabela,
    COUNT(*) AS quantidade_registros
FROM workspace.ecommerce_gold.dim_cliente

UNION ALL

SELECT
    'dim_produto' AS tabela,
    COUNT(*) AS quantidade_registros
FROM workspace.ecommerce_gold.dim_produto

UNION ALL

SELECT
    'dim_data' AS tabela,
    COUNT(*) AS quantidade_registros
FROM workspace.ecommerce_gold.dim_data

UNION ALL

SELECT
    'dim_pedido' AS tabela,
    COUNT(*) AS quantidade_registros
FROM workspace.ecommerce_gold.dim_pedido

UNION ALL

SELECT
    'dim_comercial' AS tabela,
    COUNT(*) AS quantidade_registros
FROM workspace.ecommerce_gold.dim_comercial

UNION ALL

SELECT
    'fato_item_pedido' AS tabela,
    COUNT(*) AS quantidade_registros
FROM workspace.ecommerce_gold.fato_item_pedido;


tabela,quantidade_registros
dim_cliente,25000
dim_produto,1175
dim_data,1826
dim_pedido,138116
dim_comercial,5815
fato_item_pedido,397569


**Validando a integridade entre fatos e dimensões:**

Essas consultas verificaram se a tabela possui chaves que não encontram correspondência nas dimensões.




In [0]:
%sql
SELECT COUNT(*) AS itens_sem_cliente
FROM workspace.ecommerce_gold.fato_item_pedido f
LEFT JOIN workspace.ecommerce_gold.dim_cliente c
    ON f.sk_cliente = c.sk_cliente
WHERE c.sk_cliente IS NULL;


itens_sem_cliente
0


In [0]:
%sql
SELECT COUNT(*) AS itens_sem_produto
FROM workspace.ecommerce_gold.fato_item_pedido f
LEFT JOIN workspace.ecommerce_gold.dim_produto p
    ON f.sk_produto = p.sk_produto
WHERE p.sk_produto IS NULL;


itens_sem_produto
0


In [0]:
%sql
SELECT COUNT(*) AS itens_sem_data
FROM workspace.ecommerce_gold.fato_item_pedido f
LEFT JOIN workspace.ecommerce_gold.dim_data d
    ON f.sk_data = d.sk_data
WHERE d.sk_data IS NULL;


itens_sem_data
0


In [0]:
%sql
SELECT COUNT(*) AS itens_sem_pedido
FROM workspace.ecommerce_gold.fato_item_pedido f
LEFT JOIN workspace.ecommerce_gold.dim_pedido p
    ON f.sk_pedido = p.sk_pedido
WHERE p.sk_pedido IS NULL;


itens_sem_pedido
0
